# 01 · Obtención de datos: exportaciones de café, cacao y madera (SUNAT)

**Proyecto:** Exportaciones peruanas de commodities EUDR a la Unión Europea.

**Pregunta de investigación:** ¿Cuánto exporta el Perú a la Unión Europea en café, cacao y madera (valor FOB y volumen), a qué países
se dirige y qué tan expuesto está cada producto al Reglamento (UE) 2023/1115 sobre productos libres de deforestación (EUDR)?

Este notebook descarga las fuentes oficiales y extrae solo las subpartidas que el EUDR regula para los tres productos.

| Fuente | Contenido | Uso |
|---|---|---|
| SUNAT, Nota Tributaria, **Cuadro G8** | Exportación definitiva por país de destino y subpartida nacional, mensual, 2005 a agosto 2026 | Base principal (país × subpartida × mes) |
| SUNAT, Nota Tributaria, **Cuadro G9** | Exportación definitiva por subpartida nacional, anual | Validación de totales |

Página de descarga: <https://www.sunat.gob.pe/estadisticasestudios/exportaciones.html>

Las cifras de SUNAT corresponden a **exportaciones definitivas regularizadas**, según la fecha de embarque. Los valores están en
**US$ FOB** y los pesos en **kilogramos**. Los encabezados del Excel dicen "miles de US$" y "toneladas", pero al comparar con el total
nacional se ve que las cifras están en US$ y kg (ver la validación al final).

In [1]:
import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from openpyxl import load_workbook

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ORIGINALES = RAIZ / "datos" / "originales"
CARPETA_SUNAT = ORIGINALES / "sunat_nota_tributaria"
CARPETA_SUNAT.mkdir(parents=True, exist_ok=True)

URL_BASE = "https://www.sunat.gob.pe/estadisticasestudios/nota_tributaria"
CUADROS = {
    "G8": "cdro_G8.xlsx",  # país de destino × subpartida nacional, mensual (~175 MB)
    "G9": "cdro_G9.xlsx",  # subpartida nacional, anual (~7 MB)
}

## 1. Descarga

El Cuadro G8 pesa unos 175 MB y la descarga desde SUNAT puede tardar varios minutos. Si el archivo ya existe, no se vuelve a descargar.
Este archivo **no se sube a GitHub** (supera el límite de 100 MB). Se versiona el extracto en CSV que se genera más abajo.

In [2]:
def descargar(url: str, destino: Path, forzar: bool = False) -> Path:
    if destino.exists() and destino.stat().st_size > 0 and not forzar:
        print(f"Ya existe: {destino.name} ({destino.stat().st_size / 1e6:.1f} MB)")
        return destino
    print(f"Descargando {url} ...")
    with requests.get(url, stream=True, timeout=900) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for bloque in r.iter_content(chunk_size=1 << 20):
                f.write(bloque)
    print(f"Listo: {destino.name} ({destino.stat().st_size / 1e6:.1f} MB)")
    return destino


archivos = {clave: descargar(f"{URL_BASE}/{nombre}", CARPETA_SUNAT / nombre) for clave, nombre in CUADROS.items()}

metadatos = {
    clave: {
        "url": f"{URL_BASE}/{CUADROS[clave]}",
        "archivo": str(ruta.relative_to(RAIZ)).replace("\\", "/"),
        "fecha_descarga": datetime.fromtimestamp(ruta.stat().st_mtime).strftime("%Y-%m-%d"),
        "tamano_mb": round(ruta.stat().st_size / 1e6, 1),
    }
    for clave, ruta in archivos.items()
}
(ORIGINALES / "metadatos_descarga.json").write_text(json.dumps(metadatos, indent=2, ensure_ascii=False), encoding="utf-8")
metadatos

Ya existe: cdro_G8.xlsx (175.7 MB)
Ya existe: cdro_G9.xlsx (6.9 MB)


{'G8': {'url': 'https://www.sunat.gob.pe/estadisticasestudios/nota_tributaria/cdro_G8.xlsx',
  'archivo': 'datos/originales/sunat_nota_tributaria/cdro_G8.xlsx',
  'fecha_descarga': '2026-10-31',
  'tamano_mb': 175.7},
 'G9': {'url': 'https://www.sunat.gob.pe/estadisticasestudios/nota_tributaria/cdro_G9.xlsx',
  'archivo': 'datos/originales/sunat_nota_tributaria/cdro_G9.xlsx',
  'fecha_descarga': '2026-10-31',
  'tamano_mb': 6.9}}

## 2. Alcance EUDR: qué subpartidas se extraen

El Anexo I del Reglamento (UE) 2023/1115 lista los productos regulados por código del Sistema Armonizado. En este proyecto se usan:

| Producto | Partidas SA (Anexo I EUDR) |
|---|---|
| **Café** | 0901 (café verde, tostado, descafeinado, cáscara y sucedáneos con café) |
| **Cacao** | 1801 grano · 1802 cáscara · 1803 pasta · 1804 manteca · 1805 polvo · 1806 chocolate |
| **Madera** | Capítulo 44 (madera en bruto, aserrada, perfilada, tableros, carpintería, carbón vegetal, etc.) |

Quedan fuera la pulpa y el papel (capítulos 47–49) y los muebles de madera (9401/9403), que el EUDR también regula. Esto se anota
como limitación del análisis.

In [3]:
PREFIJOS_EUDR = ("0901", "1801", "1802", "1803", "1804", "1805", "1806", "44")
MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
    "agosto": 8, "setiembre": 9, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12,
}


def es_numero(valor) -> bool:
    return isinstance(valor, (int, float)) and not isinstance(valor, bool)


def codigo_10(valor) -> str:
    """El Excel guarda la subpartida como número y pierde el cero inicial (901119000 -> 0901119000)."""
    return str(int(valor)).zfill(10)

## 3. Lectura del Cuadro G8 (país × subpartida × mes)

Cada hoja es un año. Hay filas de **país** (columna C, con el total exportado a ese país) seguidas de las filas de **subpartidas**
(columna E: código, F: descripción). Para cada mes hay tres columnas: valor FOB, peso neto y peso bruto. La hoja 2026 solo trae
los meses ya publicados (enero–agosto al momento de la descarga). Por eso los meses se leen del encabezado y no se asumen fijos.

Se guardan dos tablas:
- `extracto_g8_eudr_mensual.csv`: solo subpartidas EUDR, en formato largo (una fila por año, mes, país y subpartida).
- `extracto_g8_total_pais_mensual.csv`: el total exportado a cada país (todas las mercancías), para calcular participaciones.

In [4]:
def columnas_por_mes(encabezado) -> dict[int, int]:
    """Devuelve {mes: índice de la columna 'Valor FOB'} leyendo la fila de nombres de meses."""
    return {MESES[str(v).strip().lower()]: i for i, v in enumerate(encabezado) if v and str(v).strip().lower() in MESES}


def leer_g8(ruta: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    libro = load_workbook(ruta, read_only=True)
    productos, paises = [], []
    for hoja in libro.worksheets:
        anio = int(hoja.title)
        cols, pais = None, None
        for fila in hoja.iter_rows(values_only=True):
            if cols is None:
                if any(str(v).strip().lower() == "enero" for v in fila if v):
                    cols = columnas_por_mes(fila)
                continue
            etiqueta, codigo, descripcion = fila[2], fila[4], fila[5]
            valores = {mes: fila[i:i + 3] for mes, i in cols.items()}
            if isinstance(etiqueta, str) and etiqueta.strip() and any(es_numero(v[0]) for v in valores.values()):
                if etiqueta.strip() == "Total general":
                    continue
                pais = etiqueta.strip()
                for mes, (fob, neto, bruto) in valores.items():
                    if fob:
                        paises.append((anio, mes, pais, fob, neto or 0, bruto or 0))
            elif es_numero(codigo) and pais:
                cod = codigo_10(codigo)
                if not cod.startswith(PREFIJOS_EUDR):
                    continue
                for mes, (fob, neto, bruto) in valores.items():
                    if fob or neto:
                        productos.append((anio, mes, pais, cod, str(descripcion).strip(), fob or 0, neto or 0, bruto or 0))
        print(f"  {anio}: {len(productos):>7,} filas EUDR acumuladas · meses {min(cols)}–{max(cols)}")
    libro.close()
    columnas = ["anio", "mes", "pais_sunat", "fob_usd", "peso_neto_kg", "peso_bruto_kg"]
    df_paises = pd.DataFrame(paises, columns=columnas)
    df_productos = pd.DataFrame(productos, columns=columnas[:3] + ["subpartida", "descripcion"] + columnas[3:])
    return df_productos, df_paises


g8_eudr, g8_paises = leer_g8(archivos["G8"])
g8_eudr.shape, g8_paises.shape

  2005:   2,323 filas EUDR acumuladas · meses 1–12


  2006:   4,832 filas EUDR acumuladas · meses 1–12


  2007:   7,464 filas EUDR acumuladas · meses 1–12


  2008:  10,125 filas EUDR acumuladas · meses 1–12


  2009:  12,587 filas EUDR acumuladas · meses 1–12


...

  2023:  53,703 filas EUDR acumuladas · meses 1–12


  2024:  57,089 filas EUDR acumuladas · meses 1–12


  2025:  60,397 filas EUDR acumuladas · meses 1–12


  2026:  62,535 filas EUDR acumuladas · meses 1–8


((62535, 8), (33377, 6))

In [5]:
g8_eudr.sample(8, random_state=1)

,anio,mes,pais_sunat,subpartida,descripcion,fob_usd,peso_neto_kg,peso_bruto_kg
1173,2005,7,MEXICO,4412290000,DEMAS MADERAS CHAPADAS Q`CONT. POR LO MENOS UN...,38804.54,44450.000,4.459000e+04
11052,2009,12,COLOMBIA,0901119000,"DEMAS CAFES SIN DESCAFEINAR, SIN TOSTAR",2305398.60,1318132.844,3.646336e+07
26627,2015,5,ESTADOS UNIDOS,0901120000,"CAFE DESCAFEINADO, SIN TOSTAR",1340.00,41.484,4.212300e+01
55930,2024,3,FRANCIA,1804001100,MANTECA DE CACAO C/INDICE DE ACIDEZ EXPRESADO ...,52260.50,3577.500,3.776766e+03
11843,2009,11,FRANCIA,4420900000,"DEMAS MARQUETERIA, COFRECILLOS O ESTUCHES P`JO...",234.00,17.442,2.152000e+01
17466,2011,8,AUSTRALIA,4420100000,"ESTATUILLAS Y DEMAS OBJETOS DE ADORNO, DE MADERA",1650.00,96.286,1.085410e+02
39048,2019,12,ECUADOR,1806310000,"CHOCOLATES Y DEMAS PREPARAC. CON CACAO, EN BL...",5325.60,1749.289,1.950979e+03
24408,2014,3,COLOMBIA,1801001900,"DEMAS CACAO CRUDO EN GRANO, ENTERO O PARTIDO",291066.88,102676.000,1.030000e+05


## 4. Lectura del Cuadro G9 (subpartida, anual) para validar

El G9 trae el total anual de cada subpartida sin desagregar por país. Si el G8 está completo, la suma del G8 por subpartida y año
debe coincidir con el G9.

In [6]:
def leer_g9(ruta: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    libro = load_workbook(ruta, read_only=True)
    registros, totales = [], []
    for hoja in libro.worksheets:
        anio = int(hoja.title)
        for fila in hoja.iter_rows(values_only=True):
            if len(fila) < 10:
                continue
            if fila[2] == "Total general":
                totales.append((anio, fila[7], fila[8]))
            elif es_numero(fila[2]) and es_numero(fila[3]):
                cod = codigo_10(fila[3])
                if cod.startswith(PREFIJOS_EUDR):
                    registros.append((anio, cod, str(fila[4]).strip(), fila[5], fila[6], fila[7] or 0, fila[8] or 0, fila[9] or 0))
    libro.close()
    g9 = pd.DataFrame(registros, columns=["anio", "subpartida", "descripcion", "tipo", "sector", "fob_usd", "peso_neto_kg", "peso_bruto_kg"])
    total = pd.DataFrame(totales, columns=["anio", "fob_usd_total_peru", "peso_neto_kg_total_peru"])
    return g9, total


g9_eudr, g9_total = leer_g9(archivos["G9"])
g9_total.tail()

,anio,fob_usd_total_peru,peso_neto_kg_total_peru
16,2021,6.088208e+10,5.251225e+10
17,2022,6.348363e+10,5.553878e+10
18,2023,6.428865e+10,5.788482e+10
19,2024,7.437204e+10,6.279829e+10
20,2025,9.029908e+10,6.033535e+10


**Unidades.** El total nacional de 2024 en el G9 sale cerca de 75 000 millones. Esa es la cifra en **dólares** de las exportaciones
peruanas de ese año, no en miles de dólares. Por eso se tratan los valores como US$ y kg.

In [7]:
def familia(codigo: str) -> str:
    if codigo.startswith("0901"):
        return "Café"
    if codigo.startswith("18"):
        return "Cacao"
    return "Madera"


comparacion = (
    g8_eudr.assign(producto=g8_eudr.subpartida.map(familia)).groupby(["anio", "producto"]).fob_usd.sum().rename("fob_g8")
    .to_frame()
    .join(g9_eudr.assign(producto=g9_eudr.subpartida.map(familia)).groupby(["anio", "producto"]).fob_usd.sum().rename("fob_g9"))
)
comparacion["diferencia_%"] = (comparacion.fob_g8 / comparacion.fob_g9 - 1) * 100
comparacion.unstack("producto").round(1).tail(12)

fob_g8                                   fob_g9                \
producto         Cacao          Café       Madera         Cacao          Café   
anio                                                                            
2015      2.666355e+08  6.136827e+08  144922906.8  2.666355e+08  6.136827e+08   
2016      2.942424e+08  7.626252e+08  125290719.9  2.942424e+08  7.626252e+08   
2017      2.368189e+08  7.139012e+08  117911576.6  2.368189e+08  7.139012e+08   
2018      2.674631e+08  6.828471e+08  119321951.2  2.674631e+08  6.828471e+08   
2019      2.952700e+08  6.372958e+08  120955934.6  2.952700e+08  6.372958e+08   
2020      2.795419e+08  6.422416e+08   89931984.7  2.795419e+08  6.422416e+08   
2021      3.040134e+08  7.697333e+08  118908131.4  3.040134e+08  7.697333e+08   
2022      3.340012e+08  1.236119e+09  121790331.8  3.340012e+08  1.236119e+09   
2023      4.276978e+08  8.197098e+08   96722542.2  4.276978e+08  8.197098e+08   
2024      1.280473e+09  1.101749e+09   80260530.2  1.280473e+09  1.101749e+09   
2025      1.618274e+09  1.801967e+09   65227680.9  1.618548e+09  1.801967e+09   
2026      5.792202e+08  6.971201e+08   42798474.9           NaN           NaN   

                      diferencia_%              
producto       Madera        Cacao Café Madera  
anio                                            
2015      144922906.8          0.0  0.0    0.0  
2016      125290719.9          0.0 -0.0   -0.0  
2017      117911576.6          0.0 -0.0    0.0  
2018      119321951.2          0.0  0.0    0.0  
2019      120955934.6          0.0  0.0    0.0  
2020       89931984.7          0.0  0.0    0.0  
2021      118908131.4          0.0  0.0    0.0  
2022      121790331.8          0.0 -0.0    0.0  
2023       96722542.2          0.0 -0.0    0.0  
2024       80260530.2          0.0  0.0    0.0  
2025       65227680.9         -0.0 -0.0    0.0  
2026              NaN          NaN  NaN    NaN

Entre 2005 y 2025, la suma del G8 coincide con el G9 en los tres productos (diferencias de 0,0 %). El G9 todavía no publica 2026.
El G8 no se limita a las "principales" subpartidas y sirve como base completa por país de destino.

## 5. ¿Por qué no se usa la consulta en línea "Acumulado anual subpartida nacional/país" de Aduanet?

SUNAT ofrece una consulta web por subpartida y país (<http://www.aduanet.gob.pe/cl-ad-itestadispartida/resumenPPaisS01Alias>).
Al probarla, los totales desde 2020 están muy por debajo de las cifras oficiales. Por ejemplo, los concentrados de cobre bajan de
US$ 12 000 millones en 2019 a menos de US$ 1 000 millones. Por eso se usan los cuadros de la Nota Tributaria. La celda siguiente
reproduce la prueba (requiere conexión, y si el servidor no responde solo muestra el error).

In [8]:
URL_ADUANET = "http://www.aduanet.gob.pe/cl-ad-itestadispartida/resumenPPaisS01Alias"


def consulta_aduanet(sesion: requests.Session, anio: int, subpartida: str) -> float | None:
    datos = {"ano_prese": str(anio), "tipo_regimen": "02", "part_nandi": subpartida, "accion": "consultarResumenPPais"}
    r = sesion.post(URL_ADUANET, data=datos, timeout=30)
    r.encoding = "latin-1"
    fila_total = re.search(r"TOTAL - TOTAL</b>\s*</td>\s*<td[^>]*>\s*([\d,\.]+)\s*</td>", r.text)
    return float(fila_total.group(1).replace(",", "")) if fila_total else None


try:
    sesion = requests.Session()
    sesion.get(URL_ADUANET, timeout=30)  # la primera consulta sin sesión devuelve una página de error
    prueba = pd.DataFrame(
        [(sp, a, consulta_aduanet(sesion, a, sp)) for sp in ("0901119000", "1801001900") for a in (2019, 2024)],
        columns=["subpartida", "anio", "fob_aduanet"],
    )
    g9_todo = g9_eudr.set_index(["subpartida", "anio"]).fob_usd
    prueba["fob_nota_tributaria_g9"] = [g9_todo.get((sp, a)) for sp, a in zip(prueba.subpartida, prueba.anio)]
    prueba["aduanet_sobre_g9_%"] = (prueba.fob_aduanet / prueba.fob_nota_tributaria_g9 * 100).round(1)
    display(prueba)
except requests.RequestException as error:
    print("Aduanet no respondió:", error)

,subpartida,anio,fob_aduanet,fob_nota_tributaria_g9,aduanet_sobre_g9_%
0,0901119000,2019,6.276050e+08,6.349725e+08,98.8
1,0901119000,2024,2.606591e+08,1.099872e+09,23.7
2,1801001900,2019,1.370860e+08,1.388994e+08,98.7
3,1801001900,2024,1.407313e+07,7.398253e+08,1.9


En 2019 la consulta de Aduanet y el Cuadro G9 casi coinciden. En 2024 Aduanet reporta solo una fracción del total oficial. Pasa lo
mismo con otros productos (por ejemplo, los concentrados de cobre), así que la consulta en línea no sirve para analizar años recientes.

## 6. Guardar extractos

In [9]:
g8_eudr.to_csv(ORIGINALES / "extracto_g8_eudr_mensual.csv", index=False)
g8_paises.to_csv(ORIGINALES / "extracto_g8_total_pais_mensual.csv", index=False)
g9_eudr.to_csv(ORIGINALES / "extracto_g9_eudr_anual.csv", index=False)
g9_total.to_csv(ORIGINALES / "extracto_g9_total_peru_anual.csv", index=False)
for archivo in sorted(ORIGINALES.glob("extracto_*.csv")):
    print(f"{archivo.name:<40} {archivo.stat().st_size / 1e6:6.2f} MB")

extracto_g8_eudr_mensual.csv               7.08 MB
extracto_g8_total_pais_mensual.csv         1.90 MB
extracto_g9_eudr_anual.csv                 0.25 MB
extracto_g9_total_peru_anual.csv           0.00 MB
